# Часть 4. Выводы и ограничения

**Что в этой части:** формулируем итоги анализа и честно описываем ограничения.

> 📎 **Источник чисел.** Все статистические результаты (ANOVA, Kruskal–Wallis, Mann–Whitney, Spearman+FDR, регрессия с HC3) получены в `03_statistics.ipynb` (последний прогон) и **синхронизированы** с ним. Здесь они **интерпретируются**, а не пересчитываются.

> ⚠️ Все выводы — про **связь (ассоциацию)**, а не причинность.

## 5. Выводы

### 5.1. Общий уровень

- В анализе — **1435 постов** (без 45 свежих, младше 7 дней).
- Распределение ER **сильно перекошено вправо** (выбросы-вирусные посты), поэтому в выводах опираемся на **медианы** и всегда указываем **n** группы.
- Конкретные значения медиан/средних по каналу — из витрин `mart_*` (см. `02_visualizations.ipynb`).

### 5.2. Часть дня — утро и вечер лидируют ⏰

| Часть дня | n | медиана ER | средний ER |
|---|---|---|---|
| night (0–5) | 145 | 0.0052 | 0.0082 |
| day (12–17) | 593 | 0.0062 | 0.0088 |
| morning (6–11) | 282 | 0.0076 | 0.0119 |
| evening (18–23) | 385 | 0.0076 | 0.0091 |

*(источник: `03_statistics`, ячейка ANOVA/Kruskal)*

- Различия между **частями дня значимы**: **ANOVA F = 7.513, p = 0.00005**; **Kruskal–Wallis H = 20.294, p = 0.00015**.
- ⚠️ **Почему ANOVA по частям дня значима, а «час» в регрессии — нет.** Это **не противоречие**, а разные вопросы:
  - ANOVA/Kruskal по **4 частям дня** → p < 0.001: уровни ER между частями дня различаются;
  - ANOVA по **24 отдельным часам** → p ≈ 0.11: сигнал размывается по мелким группам;
  - линейный `hour` в регрессии → **p = 0.066** (незначим); Spearman `hour` → **p = 0.385** (незначим).
  - **Вывод:** эффект существует на уровне **«часть дня»** (утро/вечер vs ночь/день), но **не** на уровне конкретного часа и **не** линейно. Рекомендация — про **часть дня**, а не про «магический час».
- ⚠️ **Levene: дисперсии неоднородны** (p = 2.5e-05) → классическая ANOVA формально ограничена; дублируем через **Kruskal–Wallis** и опираемся на **медианы**.

### 5.3. Двойная механика ER — ключевой инсайт ⭐

Метрика **`forward_to_comment_ratio`** = (форварды / комментарии), считается **по группе** (сумма форвардов / сумма комментариев).

> ⚠️ На уровне отдельного поста при `replies = 0` отношение **не определено** (деление на ноль), поэтому метрика агрегируется по группе.

| Тип | n | форварды (ср.) | комментарии (ср.) | forward_to_comment_ratio |
|---|---|---|---|---|
| **document** | 423 | 12.75 | 7.13 | **1.79** ← репостят |
| photo | 852 | 6.88 | 5.97 | 1.15 |
| none (текст) | 160 | 4.05 | 15.79 | **0.26** ← обсуждают |

*(источник: витрина `mart_by_factor_vs_metric`, рисунок `figures/02_factor_vs_metric.png`)*

**Суть:** у `document` и `none` **похожий ER** (0.0110 vs 0.0117), но **противоположная механика** — документы **пересылают**, чистый текст **обсуждают**.

**Как выбирать формат:** охват/разлёт → документы; диалог → личный текст; массовый средний охват → фото.

### 5.4. Личные посты vs промо 📈 (с оговоркой)

- **Mann–Whitney:** медиана ER личных **0.0125** vs промо **0.0065** — различие **значимо** (U = 44275.5, **p < 0.001**; n_personal = 44, n_promo = 1361).
- **Структурная модель (HC3):** `is_personal_heur` = **+0.3592** → **≈ +43%** к ER (`exp(0.3592) − 1`), **p = 0.00116**.
- ⚠️ **Оговорка:** «личные» определены **эвристически** по лексике; `n` мал (**44 поста**) → эффект требует осторожной трактовки (вероятен конфаунд с длиной).

*(источник: `03_statistics`, Mann-Whitney + регрессия HC3)*

### 5.5. Корреляции с поправкой FDR — что реально 🎯

| feature | rho (Spearman) | значимо |
|---|---|---|
| `text_length` | 0.3600 | **да** |
| `time_since_prev_post` | 0.2365 | **да** |
| `hour` | 0.0229 | нет (p = 0.385) |
| `views` | −0.0451 | нет |

*(источник: `03_statistics`, Spearman+FDR Benjamini–Hochberg)*

**После FDR значимы 2 из 4 факторов:** длина текста и пауза между постами.

### 5.6. Структурная регрессия log(ER) — HC3 🧮

| Предиктор | coef | p (HC3) | значимо |
|---|---|---|---|
| `C(length_bucket)[medium]` | +0.3109 | 0.00049 | *** |
| `C(length_bucket)[short]` | +0.0357 | 0.774 | нет |
| `is_personal_heur` | +0.3592 | 0.00116 | ** |
| `hour` | +0.0080 | 0.066 | нет |
| `is_forwarded` | −0.0935 | 0.101 | нет |
| `is_holiday` | −0.0568 | 0.655 | нет |
| `is_pinned` | +0.7842 | 0.509 | нет |

**R² = 0.1769 | R²adj = 0.1716 | n = 1404**

*(источник: `03_statistics`, OLS с HC3; компоненты ER из предикторов исключены — устранена тавтология)*

- 📉 **Почему n = 1404, а выборка = 1435.** 31 строка выпадает из-за **NaN** в предикторах, которые нельзя логарифмировать: `log_interval` (нет предыдущего поста) и `log_len`. OLS автоматически исключает строки с NaN.
- ⚠️ `is_pinned` даёт **крупный коэффициент (+0.7842), но незначим** (p = 0.509): закреплённых постов **очень мало** → **широкий 95% CI**, выводам не доверяем.
- ⚠️ `is_forwarded` трактуется как «пост-пересылка», но семантика флага в дампе может быть шире; в модели **незначим** (p = 0.101).

> 🖼️ График коэффициентов с 95% CI — `figures/03_coefficient_plot.png`.

### 5.7. Комбинации контент × время

См. витрину `mart_by_combination` (раздел 02). Сильнейшие связки по медианному ER — **личный контент × утро/вечер**, но подгруппы личных постов **крошечные (n = 3–20)**, поэтому используем как **направление**, а не как твёрдый факт.

### Главные практические рекомендации

1. **Писать развёрнуто** — длина связана с ER сильнее всего (rho = 0.360, p_fdr ≈ 0; `length[medium]` +0.3109, p = 0.00049).
2. **Соблюдать паузы** между постами — излишняя частота снижает ER (rho = 0.2365, p_fdr ≈ 0).
3. **Публиковать важное утром/вечером** — значимый эффект на уровне **части дня** (ANOVA p = 0.00005), но **не** конкретного часа.
4. **Добавлять личные/живые посты** — эффект ≈ +43% (p = 0.001), но малая выборка (n = 44) и эвристика → осторожно.
5. **Смотреть три метрики раздельно** — единый ER скрывает механику (документы-репосты vs текст-обсуждение).

## 6. Ограничения анализа (Limitations)

1. **Выборка умеренная** (1435 постов; n регрессии = 1404). Отдельные группы крошечные (личные — 44; личные×ночь — 3), выводы на них не строим.
2. **Корреляция ≠ причинность.** Найденные связи не доказывают, что именно час или длина «создают» отклик.
3. **Предпосылки OLS нарушены:** остатки `log(ER)` ненормальны (JB p = 1.6e-04; skew = −0.273, kurt = 2.963 — умеренные), есть гетероскедастичность (BP p = 2.3e-05) → все выводы регрессии даны **с HC3**.
4. **ANOVA по частям дня ограничена** неоднородностью дисперсий (Levene p = 2.5e-05) → дублируем **Kruskal–Wallis**, опираемся на **медианы**.
5. **Эвристика личных постов** (`is_personal_heur`) даёт ложные срабатывания и пропуски. Ручная разметка и оценка precision/recall не выполнялись.
6. **`is_forwarded` неоднозначен** — семантика флага в дампе может быть шире; в модели незначим (p = 0.101).
7. **Просмотры — прокси охвата**, не уникальные пользователи (Telegram их не отдаёт).
8. **«Дозревающие» метрики** учтены окном 7 дней; даже 7-дневные посты могут добирать просмотры.
9. **Выбросы** (тяжёлый правый хвост ER) влияют на средние → смотрим и медиану, и размер группы (n).

**По данным:**
- Данные публичны, собраны через официальный публичный API Telegram (Telethon).
- Сырые персональные данные не публикуются, читатели не идентифицируются.
- Анализ не претендует на маркетинговое таргетирование.

## 7. Что дальше

- 2–3 главных инсайта с числами → в письмо (`letter/letter_to_kryli.md`).
- (опц.) Экспорт графиков (png) в `notebooks/figures/`.
- (опц.) Финальный прогон `sql/data_quality.sql` перед публикацией.

> 📎 При пересчёте `03_statistics.ipynb` — обновить числа здесь и в `conclusions.md` (единый источник истины — `03`).